In [13]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
from scipy.stats import kruskal
import re
import os

warnings.filterwarnings('ignore')

#plt.rcParams['font.family'] = 'AppleGothic'    # Mac
plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [14]:
DATA_PATH = "../../../../data/preprocessed/tableau_allgames_prelaunch_source.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns.tolist())
df.head()

(17648, 44)
['appid', 'game_name', 'condition_type', 'condition_type_kor', 'condition_value', 'genres_text', 'price_group', 'top_steam_tags_text', 'categories_text', 'play_style', 'issue_name_kor', 'issue_review_count', 'positive_review_count', 'negative_review_count', 'mixed_review_count', 'high_urgency_review_count', 'positive_present', 'negative_present', 'mixed_present', 'high_urgency_present', 'both_positive_negative_present', 'review_count', 'llm_positive_count', 'llm_negative_count', 'llm_mixed_count', 'high_urgency_count', 'llm_positive_ratio', 'llm_negative_ratio', 'high_urgency_ratio', 'condition_game_count', 'issue_game_count', 'issue_game_ratio', 'positive_game_count', 'negative_game_count', 'mixed_game_count', 'high_urgency_game_count', 'both_positive_negative_game_count', 'priority_level', 'priority_order', 'tag_dna_game_count', 'top_performance_game_count', 'top_performance_ratio', 'avg_positive_rate', 'median_total_reviews']


,appid,game_name,condition_type,condition_type_kor,condition_value,genres_text,price_group,top_steam_tags_text,categories_text,play_style,...,mixed_game_count,high_urgency_game_count,both_positive_negative_game_count,priority_level,priority_order,tag_dna_game_count,top_performance_game_count,top_performance_ratio,avg_positive_rate,median_total_reviews
0,571740,Golf It!,price_group,가격대,0-5,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,...,0,8,1,상,1,NaN,NaN,NaN,NaN,NaN
1,571740,Golf It!,play_style,플레이 방식,멀티/협동 요소 포함,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,...,1,11,4,상,1,NaN,NaN,NaN,NaN,NaN
2,571740,Golf It!,genre,장르,Casual,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,...,0,20,7,상,1,NaN,NaN,NaN,NaN,NaN
3,571740,Golf It!,genre,장르,Indie,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,...,1,46,17,상,1,NaN,NaN,NaN,NaN,NaN
4,571740,Golf It!,genre,장르,Simulation,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,...,0,14,3,상,1,NaN,NaN,NaN,NaN,NaN


In [15]:
df_raw = df.copy()

In [16]:
SAVE_DIR = "../../../../data/preprocessed/tableau_output"

os.makedirs(SAVE_DIR, exist_ok=True)

In [17]:
import pandas as pd
import os

# =========================================================
# 01. 파일 불러오기
# =========================================================
DATA_PATH = "../../../../data/preprocessed/tableau_allgames_prelaunch_source.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
print(df.columns.tolist())
df.head()

# =========================================================
# 02. 원본 복사
# =========================================================

df_raw = df.copy()


# =========================================================
# 03. 저장 폴더 만들기
# =========================================================

SAVE_DIR = "../../../../data/preprocessed/tableau_output"

os.makedirs(SAVE_DIR, exist_ok=True)

# =========================================================
# 04. 기본 확인
# =========================================================

print("행 개수:", len(df_raw))
print("게임 수:", df_raw["appid"].nunique())

print(df_raw["condition_type"].value_counts())
print(df_raw["condition_type_kor"].value_counts())


# =========================================================
# 05. 전체 KPI용 base 파일 만들기
# =========================================================

base_cols = [
    "appid",
    "game_name",
    "genres_text",
    "price_group",
    "top_steam_tags_text",
    "categories_text",
    "play_style",
    "review_count",
    "llm_positive_count",
    "llm_negative_count",
    "llm_mixed_count",
    "high_urgency_count",
    "llm_positive_ratio",
    "llm_negative_ratio",
    "high_urgency_ratio",
    "avg_positive_rate",
    "median_total_reviews"
]

base_cols = [col for col in base_cols if col in df_raw.columns]

tableau_base = (
    df_raw[base_cols]
    .drop_duplicates(subset=["appid"])
    .reset_index(drop=True)
)

print("base 파일 크기:", tableau_base.shape)
display(tableau_base.head())

tableau_base.to_csv(
    os.path.join(SAVE_DIR, "tableau_base_game_level.csv"),
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 06. condition_type별 파일 저장 함수
# =========================================================

def save_condition_file(df, condition_type, file_name):
    """
    condition_type 기준으로 Tableau용 파일을 저장하는 함수

    condition_type 예:
    - genre
    - steam_tag
    - category
    - play_style
    - price_group
    """

    temp = df[df["condition_type"] == condition_type].copy()

    temp = temp.reset_index(drop=True)

    save_path = os.path.join(SAVE_DIR, file_name)

    temp.to_csv(
        save_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"{condition_type} 저장 완료:", temp.shape)
    print(save_path)

    return temp


# =========================================================
# 07. Tableau용 분석 파일 저장
# =========================================================

tableau_genre = save_condition_file(
    df_raw,
    condition_type="genre",
    file_name="tableau_genre_issue_long.csv"
)

tableau_steam_tag = save_condition_file(
    df_raw,
    condition_type="steam_tag",
    file_name="tableau_steam_tag_issue_long.csv"
)

tableau_category = save_condition_file(
    df_raw,
    condition_type="category",
    file_name="tableau_category_issue_long.csv"
)

tableau_play_style = save_condition_file(
    df_raw,
    condition_type="play_style",
    file_name="tableau_play_style_issue_long.csv"
)

tableau_price_group = save_condition_file(
    df_raw,
    condition_type="price_group",
    file_name="tableau_price_group_issue_long.csv"
)


# =========================================================
# 08. 저장 결과 확인
# =========================================================

print(os.listdir(SAVE_DIR))

(17648, 44)
['appid', 'game_name', 'condition_type', 'condition_type_kor', 'condition_value', 'genres_text', 'price_group', 'top_steam_tags_text', 'categories_text', 'play_style', 'issue_name_kor', 'issue_review_count', 'positive_review_count', 'negative_review_count', 'mixed_review_count', 'high_urgency_review_count', 'positive_present', 'negative_present', 'mixed_present', 'high_urgency_present', 'both_positive_negative_present', 'review_count', 'llm_positive_count', 'llm_negative_count', 'llm_mixed_count', 'high_urgency_count', 'llm_positive_ratio', 'llm_negative_ratio', 'high_urgency_ratio', 'condition_game_count', 'issue_game_count', 'issue_game_ratio', 'positive_game_count', 'negative_game_count', 'mixed_game_count', 'high_urgency_game_count', 'both_positive_negative_game_count', 'priority_level', 'priority_order', 'tag_dna_game_count', 'top_performance_game_count', 'top_performance_ratio', 'avg_positive_rate', 'median_total_reviews']
행 개수: 17648
게임 수: 151
condition_type
steam_ta

,appid,game_name,genres_text,price_group,top_steam_tags_text,categories_text,play_style,review_count,llm_positive_count,llm_negative_count,llm_mixed_count,high_urgency_count,llm_positive_ratio,llm_negative_ratio,high_urgency_ratio,avg_positive_rate,median_total_reviews
0,571740,Golf It!,"Casual, Indie, Simulation, Sports",0-5,"Multiplayer, Mini Golf, Golf, Casual, Sports, ...","Single-player, Multi-player, PvP, Online PvP, ...",멀티/협동 요소 포함,48,29,15,2,9,60.4,31.2,18.8,NaN,NaN
1,603390,Stone Story RPG,"Adventure, Casual, Indie, RPG",20-40,"RPG, Adventure, Casual, Indie, Programming, Re...","Single-player, Steam Achievements, Camera Comf...",Single-player 중심,7,7,0,0,0,100.0,0.0,0.0,NaN,NaN
2,638990,"UNDYING - ""KINGDOM""","Adventure, Indie",10-20,"Story Rich, Multiple Endings, Action RPG, Surv...","Single-player, Steam Achievements, Full contro...",Single-player 중심,30,20,4,6,6,66.7,13.3,20.0,NaN,NaN
3,695330,SEASON: A letter to the future,"Adventure, Casual, Indie",10-20,"Open World, Mystery, Visual Novel, Narrative, ...","Single-player, Steam Achievements, Full contro...",Single-player 중심,99,87,6,6,6,87.9,6.1,6.1,NaN,NaN
4,824600,HROT,"Action, Indie",10-20,"Retro, FPS, Shooter, Boomer Shooter, Indie, Fi...","Single-player, Steam Achievements, Full contro...",Single-player 중심,100,73,11,14,7,73.0,11.0,7.0,NaN,NaN


genre 저장 완료: (3767, 44)
../../../../data/preprocessed/tableau_output\tableau_genre_issue_long.csv
steam_tag 저장 완료: (11561, 44)
../../../../data/preprocessed/tableau_output\tableau_steam_tag_issue_long.csv
category 저장 완료: (0, 44)
../../../../data/preprocessed/tableau_output\tableau_category_issue_long.csv
play_style 저장 완료: (1160, 44)
../../../../data/preprocessed/tableau_output\tableau_play_style_issue_long.csv
price_group 저장 완료: (1160, 44)
../../../../data/preprocessed/tableau_output\tableau_price_group_issue_long.csv
['tableau_base_game_level.csv', 'tableau_category_issue_long.csv', 'tableau_genre_issue_long.csv', 'tableau_play_style_issue_long.csv', 'tableau_price_group_issue_long.csv', 'tableau_steam_tag_issue_long.csv']
